In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Hydrology_Applications/Climate_change_analysis


In [2]:
!pip install fsspec s3fs dask
!pip install icechunk xarray

In [29]:
import icechunk
import xarray as xr

# Open the store (free, anonymous)
storage = icechunk.s3_storage(
    bucket="earthmover-icechunk-era5",
    prefix="icechunkV2",
    region="us-east-1",
    anonymous=True,
)
repo = icechunk.Repository.open(storage)
session = repo.readonly_session("main")

ds = xr.open_zarr(
    session.store,
    group="single/spatial",   # one full global map per hour
    consolidated=False,
    chunks={}
)

t2m = ds["t2m"].sel(valid_time=slice("1985-01-01", "2014-12-31")
)

# Compute daily min and max
tmin = t2m.resample(valid_time="1D").min() - 273.15
tmax = t2m.resample(valid_time="1D").max() - 273.15

KeyboardInterrupt: 

In [ ]:
tp = ds["tp"].sel(valid_time=slice("1985-01-01", "2014-12-31"))
tp = tp.resample(valid_time="1D").sum()
tp

In [3]:
import os
import icechunk
import numpy as np
import xarray as xr
import geopandas as gpd
from pathlib import Path

# =============================================================
# CONFIG
# =============================================================
BASE_PATH = "/content/drive/MyDrive/NEX_GDDP_CMIP6"
SHP_PATH  = "Basin_shp/Karnali_basin.shp"
OUT_DIR   = "/content/drive/MyDrive/BC_Output"

MODELS    = ["ACCESS-CM2", "ACCESS-ESM1-5"]
SCENARIOS = ["historical", "ssp245", "ssp585"]

os.makedirs(OUT_DIR, exist_ok=True)

# =============================================================
# STEP 1: Karnali bounding box
# =============================================================
basin = gpd.read_file(SHP_PATH).to_crs("EPSG:4326")
minx, miny, maxx, maxy = basin.total_bounds
lon_min, lon_max = minx - 0.25, maxx + 0.25
lat_min, lat_max = miny - 0.25, maxy + 0.25

# =============================================================
# STEP 2: Load ERA5 from icechunk S3 (free, anonymous)
# =============================================================
print("Connecting to ERA5 icechunk store...")
storage = icechunk.s3_storage(
    bucket="earthmover-icechunk-era5",
    prefix="icechunkV2",
    region="us-east-1",
    anonymous=True,
)
repo    = icechunk.Repository.open(storage)
session = repo.readonly_session("main")

ds_era5 = xr.open_zarr(
    session.store,
    group="single/spatial",
    consolidated=False,
    chunks={"valid_time": 24},
)

# --- Precipitation (tp): m/hour → mm/day ---
tp = ds_era5["tp"].sel(
    valid_time=slice("1985-01-01", "2014-12-31"),
    latitude=slice(lat_max, lat_min),   # ERA5 is N→S
    longitude=slice(lon_min, lon_max),
)
era5_pr = tp.resample(valid_time="1D").sum() * 1000   # m → mm/day

# --- Temperature (t2m): K → °C, daily min/max ---
t2m = ds_era5["t2m"].sel(
    valid_time=slice("1985-01-01", "2014-12-31"),
    latitude=slice(lat_max, lat_min),
    longitude=slice(lon_min, lon_max),
)
era5_tmin = t2m.resample(valid_time="1D").min() - 273.15
era5_tmax = t2m.resample(valid_time="1D").max() - 273.15

# Rename coords to match CMIP6
era5_pr   = era5_pr.rename({"valid_time": "time", "latitude": "lat", "longitude": "lon"})
era5_tmin = era5_tmin.rename({"valid_time": "time", "latitude": "lat", "longitude": "lon"})
era5_tmax = era5_tmax.rename({"valid_time": "time", "latitude": "lat", "longitude": "lon"})

print(f"ERA5 loaded: {era5_pr.sizes}")

Connecting to ERA5 icechunk store...
ERA5 loaded: Frozen({'time': 10957, 'lat': 11, 'lon': 14})


In [ ]:
# =============================================================
# STEP 3: Linear Scaling bias correction
# =============================================================
def bias_correct(da_cmip, era5_ref, kind="ratio"):
    """
    Regrid CMIP6 → ERA5 grid, then apply monthly linear scaling.
    kind = 'ratio'      for precipitation (multiplicative)
    kind = 'difference' for temperature   (additive)
    """
    # Regrid to ERA5 grid
    da = da_cmip.interp(lat=era5_ref["lat"], lon=era5_ref["lon"], method="nearest")

    # Monthly correction factors (1985-2014 overlap only)
    da_hist_ov = da.sel(time=slice("1985-01-01", "2014-12-31"))
    obs_mon    = era5_ref.groupby("time.month").mean()
    sim_mon    = da_hist_ov.groupby("time.month").mean()

    if kind == "ratio":
        factors = (obs_mon / sim_mon.clip(min=1e-6)).clip(0.1, 10.0)
    else:  # additive
        factors = obs_mon - sim_mon

    # Apply factor month-by-month
    month_idx    = da["time"].dt.month
    fac          = factors.sel(month=month_idx).drop_vars("month")
    fac["time"]  = da["time"]

    if kind == "ratio":
        return (da * fac).clip(min=0)
    else:
        return da + fac


# =============================================================
# STEP 4: Loop over models & scenarios
# =============================================================
for model in MODELS:
    print(f"\n=== {model} ===")

    for scen in SCENARIOS:
        print(f"  {scen}...", end=" ")
        scen_path = Path(BASE_PATH, model, scen)

        results = {}

        for var, era5_ref, kind in [
            ("pr",   era5_pr,   "ratio"),
            ("tasmin", era5_tmin, "difference"),
            ("tasmax", era5_tmax, "difference"),
        ]:
            nc_files = sorted(scen_path.glob(f"{var}_*.nc"))
            if not nc_files:
                continue

            da = xr.open_mfdataset(nc_files, combine="by_coords")[var]
            da = da.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

            # Unit conversions
            if var == "pr":
                da = da * 86400          # kg/m2/s → mm/day
            else:
                da = da - 273.15         # K → °C

            results[f"{var}_bc"] = bias_correct(da, era5_ref, kind)

        if not results:
            print("no files, skipping")
            continue

        # Save all variables in one file
        out_ds   = xr.Dataset(results)
        out_path = Path(OUT_DIR) / f"{model}_{scen}_bc.nc"
        out_ds.to_netcdf(out_path)
        print(f"saved → {out_path}")

print("\nDone!")

In [ ]:
# =============================================================
# METHOD 1 — LINEAR SCALING
# =============================================================
# Principle:
#   Precipitation  → multiply by monthly mean ratio  (obs/sim)
#   Temperature    → add monthly mean difference     (obs−sim)
# Calibrated on 1985-2014; same factor applied to all scenarios.
# =============================================================

def linear_scaling(da, era5_ref, kind="ratio"):
    """
    Linear Scaling bias correction.

    Precipitation (ratio / multiplicative):
        Step 1 : resample daily → monthly sums  (mm/month)
        Step 2 : long-term monthly mean          (mean of all Jan, Feb … Dec)
        Step 3 : factor = mean_ERA5[m] / mean_CMIP6_hist[m]
        Step 4 : apply factor to every time step by its calendar month

    Temperature (difference / additive):
        Step 1 : resample daily → monthly means  (°C/month)
        Step 2 : long-term monthly mean
        Step 3 : delta = mean_ERA5[m] − mean_CMIP6_hist[m]
        Step 4 : add delta to every time step by its calendar month
    """
    # ── 0. Regrid CMIP6 → ERA5 grid ──────────────────────────
    da = da.interp(lat=era5_ref["lat"], lon=era5_ref["lon"], method="nearest")

    # ── historical overlap period ─────────────────────────────
    da_hist  = da.sel(time=slice("1985-01-01", "2014-12-31"))

    if kind == "ratio":
        # Step 1: daily → monthly total  (mm/month)
        obs_monthly = era5_ref.resample(time="1MS").sum()   # ERA5
        sim_monthly = da_hist.resample(time="1MS").sum()    # CMIP6 hist

        # Step 2: long-term mean of each calendar month
        obs_mon_mean = obs_monthly.groupby("time.month").mean()  # shape (12, lat, lon)
        sim_mon_mean = sim_monthly.groupby("time.month").mean()

        # Step 3: ratio factor per calendar month
        factors = (obs_mon_mean / sim_mon_mean.clip(min=1e-6)).clip(0.1, 10.0)

    else:  # temperature — additive
        # Step 1: daily → monthly mean  (°C/month)
        obs_monthly = era5_ref.resample(time="1MS").mean()
        sim_monthly = da_hist.resample(time="1MS").mean()

        # Step 2 & 3: long-term monthly mean difference
        obs_mon_mean = obs_monthly.groupby("time.month").mean()
        sim_mon_mean = sim_monthly.groupby("time.month").mean()
        factors = obs_mon_mean - sim_mon_mean

    # ── Step 4: apply factor to full time series ──────────────
    month_idx   = da["time"].dt.month
    fac         = factors.sel(month=month_idx).drop_vars("month")
    fac["time"] = da["time"]

    if kind == "ratio":
        return (da * fac).clip(min=0)
    else:
        return da + fac


# =============================================================
# METHOD 2 — QUANTILE MAPPING
# =============================================================
# Principle:
#   For each grid cell, build empirical CDFs from ERA5 (obs)
#   and CMIP6 historical simulation over 1985-2014.
#   Map every future value through:
#       x_corrected = CDF_obs⁻¹( CDF_sim(x) )
#   i.e. find which percentile the raw value sits at in the
#   simulated distribution, then replace it with the ERA5
#   value at that same percentile.
# =============================================================

def quantile_mapping_1d(raw, sim_hist, obs_hist, n_q=100):
    """
    Apply QM to a 1-D time series.
    raw      : values to correct (future or historical)
    sim_hist : CMIP6 historical values used to build sim CDF
    obs_hist : ERA5 historical values used to build obs CDF
    """
    quantiles = np.linspace(0, 100, n_q)

    sim_q = np.nanpercentile(sim_hist, quantiles)
    obs_q = np.nanpercentile(obs_hist, quantiles)

    # Build transfer function: sim quantile → obs quantile
    transfer = interp1d(sim_q, obs_q,
                        bounds_error=False,
                        fill_value=(obs_q[0], obs_q[-1]))
    return transfer(raw)


def quantile_mapping(da, era5_ref, n_q=100):
    """
    Apply QM grid-cell by grid-cell.
    da       : full DataArray (all scenarios/years) after regridding
    era5_ref : ERA5 reference DataArray (1985-2014)
    """
    da = da.interp(lat=era5_ref["lat"], lon=era5_ref["lon"], method="nearest")
    da_hist = da.sel(time=slice("1985-01-01", "2014-12-31"))

    # Apply QM using xarray's apply_ufunc (vectorised over lat/lon)
    corrected = xr.apply_ufunc(
        quantile_mapping_1d,
        da,                    # raw values to correct
        da_hist,               # sim historical (passed as dask-compatible arg)
        era5_ref,              # obs historical
        input_core_dims=[["time"], ["time"], ["time"]],
        output_core_dims=[["time"]],
        kwargs={"n_q": n_q},
        vectorize=True,        # loop over lat/lon
        dask="parallelized",
        output_dtypes=[float],
    )
    # apply_ufunc moves "time" to last axis; transpose back
    corrected = corrected.transpose(*da.dims)
    corrected["time"] = da["time"]
    return corrected.clip(min=0)


# =============================================================
# MAIN LOOP
# =============================================================
for model in MODELS:
    print(f"\n=== {model} | method={METHOD} ===")

    for scen in SCENARIOS:
        print(f"  {scen}...", end=" ")
        scen_path = Path(BASE_PATH, model, scen)
        results   = {}

        for var, era5_ref, kind in [
            ("pr",     era5_pr,   "ratio"),
            ("tasmin", era5_tmin, "difference"),
            ("tasmax", era5_tmax, "difference"),
        ]:
            nc_files = sorted(scen_path.glob(f"{var}_*.nc"))
            if not nc_files:
                continue

            da = xr.open_mfdataset(nc_files, combine="by_coords")[var]
            da = da.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
            da = da * 86400 if var == "pr" else da - 273.15   # unit conversion

            if METHOD == "LS":
                results[f"{var}_bc"] = linear_scaling(da, era5_ref, kind)
            else:
                results[f"{var}_bc"] = quantile_mapping(da, era5_ref, N_QUANTILES)

        if not results:
            print("no files, skipping")
            continue

        out_path = Path(OUT_DIR) / f"{model}_{scen}_{METHOD}_bc.nc"
        xr.Dataset(results).to_netcdf(out_path)
        print(f"saved → {out_path}")

print("\nDone!")